# Reasoning SLM - Stage 3: GRPO

## Introduction

Stage 3 starts from the Stage 2 SFT model and trains it with GRPO on the same GSM8K train problems. Nothing imitates a reference solution: the model samples several answers per problem, and the answers the checker marks correct are reinforced.

- Starting point: the merged SFT model, with a fresh LoRA adapter for GRPO.
- Rewards: correctness (1/0, the same cut and extractor as eval) plus a small format bonus (0.1) for a final `#### <number>` line.
- A short smoke run (100 steps) first, to check that the reward rises before a long run.
- The same eval as Stages 1 and 2: the 499-problem validation subset, greedy decoding, at most 512 new tokens, the same stop rules, and the 20-prompt general check.

**Target from Stage 2:** SFT fixed the format but stayed at 46.5%, while Qwen's instruct model solves about 61.5% with the format fixed. Can GRPO move the model from about 47% toward about 61%?

## Setup

Clone the repository and install the requirements, then install vLLM, which GRPO uses to sample completions. vLLM is installed here only, not in `requirements.txt`: it pins its own torch version, while Stages 1 and 2 run on Colab's default one. TRL 1.16 supports vLLM 0.22 to 0.31. Restart the runtime once after the install, then run the notebook from the top.

The SFT model, checkpoints, the adapter and the merged model live on Google Drive (`/content/drive/MyDrive/slm-post-training`). Results are written to `results/stage3/`. Download them after each run, because `/content` is wiped when the runtime disconnects.

In [ ]:
import os

REPO_URL = "https://github.com/JuaniLlaberia/slm-post-training.git"
REPO_DIR = "/content/slm-post-training"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
!pip install -q "vllm>=0.22,<0.32"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

### Imports

In [ ]:
from src.config import RESULTS_DIR, SEED
from src.data.gsm8k import load_gsm8k
from src.evaluation.general_check import GENERAL
from src.evaluation.generation import generate
from src.evaluation.harness import evaluate, save_results
from src.prompts.few_shot import STOP, build_prompt
from src.training.merge import merge_adapter
from src.training.rewards import correctness_reward, format_reward

from peft import LoraConfig
from transformers import AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

DRIVE_DIR = "/content/drive/MyDrive/slm-post-training"
SFT_MODEL = f"{DRIVE_DIR}/models/sft-merged"
STAGE_DIR = RESULTS_DIR / "stage3"

## Data

The same train split as Stages 1 and 2 (6,974 problems), with a different shape. GRPO needs only the prompt and the gold answer, no solution, since the model generates its own answers. The prompt is the zero-shot `Question: … Answer:` prompt used for SFT. Reusing the questions SFT already saw is standard: in RL the model learns from its own sampled answers, not from the reference solutions.

The validation subset stays out of training and is only used to evaluate afterwards.

In [ ]:
train_dataset, val_dataset, test_dataset = load_gsm8k()

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

In [ ]:
def to_grpo(ex):
    """
    Turns a GSM8K example into a GRPO row: the zero-shot prompt and the gold answer, no solution.
    """
    return {"prompt": build_prompt(ex["question"]),
            "gold": ex["gold"]}

grpo_train = train_dataset.map(to_grpo, remove_columns=train_dataset.column_names)
grpo_train

## Rewards

`correctness_reward` gives 1.0 when the extracted answer equals the gold, else 0.0. GRPO sampling only stops at EOS or the token limit, so the reward first cuts each completion at `\nQuestion:`, as `generate()` does at eval time, then applies the same `answer_extractor`: the RL checker and the eval checker agree. `format_reward` adds 0.1 when the completion ends with a `#### <number>` line. Both live in `src/training/rewards.py`, and TRL passes the dataset's `gold` column to them.

As a check, scoring the saved Stage 2 outputs with the rewards should reproduce the Stage 2 eval exactly.

In [ ]:
import json

with open(RESULTS_DIR / "stage2" / "sft_0shot_val500.json", encoding="utf-8") as file:
    sft_rows = json.load(file)

outputs = [r["output"] for r in sft_rows]
correct = correctness_reward(outputs, [r["gold"] for r in sft_rows])
formatted = format_reward(outputs)

print(f"correctness: {sum(correct) / len(correct):.1%}  (Stage 2 eval: 46.5%)")
print(f"format: {sum(f > 0 for f in formatted) / len(formatted):.1%}")

The rewards reproduce the Stage 2 eval (46.5%). 98.4% of the SFT outputs already end with a `####` line, so the format bonus is close to constant: it guards the format rather than teaching it.

## Model

The merged SFT model is the starting point, and GRPO trains a fresh LoRA adapter on top: rank 16, alpha 32, every linear layer, as in Stage 2, but without dropout, which would add noise to the log-probabilities GRPO compares. With `beta > 0`, the reference model for the KL penalty is the same model with the adapter disabled (the SFT model), so it costs no extra memory.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL)

RANK = 16
ALPHA = 32

lora_config = LoraConfig(
    r=RANK,
    lora_alpha=ALPHA,
    target_modules="all-linear",
    lora_dropout=0.0,
    bias="none",
)

## Training

The smoke run: 100 steps at learning rate 5e-6, much lower than SFT, because RL is easy to destabilize. Each step samples 8 completions for each of 8 prompts (16 completions per batch × 4 accumulation steps) at temperature 0.8: GRPO needs varied answers to compare, so sampling is not greedy. Completions are limited to 512 tokens, as in eval, and vLLM generates them on the same GPU as training.

The parameters still to decide for the full run are set explicitly, so the config records them:

| Parameter | Smoke run | What it controls |
| --- | --- | --- |
| `beta` | 0.0 | KL penalty toward the SFT model. At 0, TRL doesn't compute or log `kl` |
| `num_generations` | 8 | Answers per prompt compared in each group |
| `loss_type` | `"dapo"` | How token losses are averaged (TRL 1.16 default) |
| `scale_rewards` | `"group"` | Whether advantages are divided by the group's reward std (TRL 1.16 default) |
| `learning_rate` | 5e-6 | Step size |
| `temperature` | 0.8 | How varied the sampled answers are |
| `max_steps` | 100 | Run length; the checkpoint is chosen on validation |

What the smoke run should show before a long run:

- `rewards/correctness_reward/mean` trending up. It starts at the SFT model's accuracy at temperature 0.8, which can be below the greedy 46.5%.
- `frac_reward_zero_std` well below 1: a group where all 8 answers get the same reward carries no learning signal.
- `completions/mean_length` and `completions/clipped_ratio` stable: no runaway length or repetition.
- `entropy` not collapsing toward 0, and `grad_norm` without spikes.

In [ ]:
RUN_NAME = "grpo_smoke"

args = GRPOConfig(
    output_dir=f"{DRIVE_DIR}/grpo/{RUN_NAME}",
    max_steps=100,                    # smoke run; the real run comes after the reading
    learning_rate=5e-6,               # much lower than SFT: RL is easy to destabilize
    per_device_train_batch_size=16,   # completions per batch; must be a multiple of num_generations
    gradient_accumulation_steps=4,    # 64 completions per step -> 8 prompts per step
    num_generations=8,
    max_completion_length=512,
    temperature=0.8,                  # sampling, NOT greedy: GRPO needs varied answers to compare
    beta=0.0,                         # KL weight; to decide for the full run
    loss_type="dapo",
    scale_rewards="group",
    use_vllm=True,
    vllm_mode="colocate",             # vLLM on the same GPU as training
    vllm_max_model_length=1024,       # longest prompt + 512 completion tokens fits
    bf16=True,
    logging_steps=5,
    save_steps=50,
    seed=SEED,
    report_to="none",
)

trainer = GRPOTrainer(
    model=SFT_MODEL,
    reward_funcs=[correctness_reward, format_reward],
    args=args,
    train_dataset=grpo_train,
    peft_config=lora_config,
    processing_class=tokenizer,
)

print({k: getattr(args, k) for k in ["beta", "num_generations", "loss_type", "scale_rewards",
                                     "learning_rate", "temperature", "max_steps"]})

In [ ]:
trainer.train()

In [ ]:
import pandas as pd

logs = pd.DataFrame(trainer.state.log_history)
cols = ["step", "rewards/correctness_reward/mean", "rewards/format_reward/mean", "frac_reward_zero_std",
        "completions/mean_length", "completions/clipped_ratio", "entropy", "grad_norm", "kl"]
print(logs[[c for c in cols if c in logs]].dropna(subset=["rewards/correctness_reward/mean"]).to_string(index=False))

### Saving

As in Stage 2: the LoRA adapter alone, and a merged standalone model that `evaluate` loads like any Hugging Face model. The adapter is saved first, because merging removes it from `trainer.model`. GRPO has no eval loss to pick a best checkpoint, so this is the model at the last step.

In [ ]:
trainer.save_model(f"{DRIVE_DIR}/adapters/{RUN_NAME}")

GRPO_MODEL = f"{DRIVE_DIR}/models/{RUN_NAME}-merged"
merged = trainer.model.merge_and_unload()
merged.save_pretrained(GRPO_MODEL)
tokenizer.save_pretrained(GRPO_MODEL)

## GRPO Model Eval

The merged model is scored zero-shot (`Question: … Answer:`) with the Stage 1 harness and settings, on the validation subset kept out of training.

In [ ]:
questions = val_dataset["question"]
golds = val_dataset["gold"]

grpo_rows = evaluate(GRPO_MODEL, lambda q, _: build_prompt(q), questions, golds, stop=STOP)

In [ ]:
save_results(grpo_rows, STAGE_DIR / f"{RUN_NAME}_0shot_val500.json")

### Checkpoint Choice

Training checkpoints hold only the adapter. `merge_adapter` merges each one into the SFT model, and the harness scores it on validation, so the run length is chosen by validation accuracy. The test set stays untouched until the final numbers.

In [ ]:
import glob

for ckpt in sorted(glob.glob(f"{args.output_dir}/checkpoint-*"), key=lambda p: int(p.split("-")[-1])):
    step = ckpt.split("-")[-1]
    path = merge_adapter(SFT_MODEL, ckpt, "/content/checkpoint-merged")
    rows = evaluate(path, lambda q, _: build_prompt(q), questions, golds, stop=STOP)
    save_results(rows, STAGE_DIR / f"{RUN_NAME}_step{step}_0shot_val500.json")

## General Check

The same 20 prompts and pass rules as Stages 1 and 2 (content and clean), with the zero-shot `Question: … Answer:` prompt.

In [ ]:
general_results = generate(GRPO_MODEL, lambda q, _: build_prompt(q), [g["prompt"] for g in GENERAL], stop=STOP, max_new_tokens=256)

In [ ]:
save_results(general_results, STAGE_DIR / f"{RUN_NAME}_general.json")

In [ ]:
general_results

**GRPO model, zero-shot**

| ID | Category | Content | Clean | Note |
| --- | --- | --- | --- | --- |
| w1 | writing | | | |
| w2 | writing | | | |
| w3 | writing | | | |
| w4 | writing | | | |
| f1 | format | | | |
| f2 | format | | | |
| f3 | format | | | |
| f4 | format | | | |
| t1 | translation | | | |
| t2 | translation | | | |
| k1 | facts | | | |
| k2 | facts | | | |
| k3 | facts | | | |
| k4 | facts | | | |
| r1 | rewrite | | | |
| r2 | rewrite | | | |
| r3 | rewrite | | | |
| j1 | judgment | | | |
| j2 | judgment | | | |
| j3 | judgment | | | |
| **Total** | | **/20** | **/20** | |

## Stage 3 Summary

Validation subset, 499 problems (±2.2 pts).

| Model | Prompt | Accuracy | No-answer | General: content | General: clean |
| --- | --- | --- | --- | --- | --- |
| Base | zero-shot | 0.2% (48.3% lenient) | 95.2% | 16/20 | 6/20 |
| Base | 8-shot | 46.9% | 4.6% | - | - |
| Base + SFT | zero-shot | 46.5% | 1.2% | 8/20 | 3/20 |
| Base + SFT + GRPO | zero-shot | | | | |
| Instruct | chat, zero-shot | 53.3% (61.5% lenient) | 22.4% | 17/20 | 16/20 |